# Experiment: SpectralBio Final Accept Hardening Part 6 (L4)

This notebook is the breadth-and-failure-analysis notebook.

## Scope
- Expand the support-ranked panel from 10 genes to top 25 feasible genes
- Score the panel under 150M and, if feasible, 650M shadow mode
- Run BRCA1 full-set failure analysis by domain, review status, substitution class, and confidence strata
- Generate tables and figures that convert BRCA1 from a naked weakness into explained heterogeneity

## Intended runtime
- Target hardware: L4
- Optional A100 rerun if we decide to push 650M across the full top-25 panel


In [ ]:
from pathlib import Path

USE_GOOGLE_DRIVE = False
DRIVE_MOUNT_POINT = Path('/content/drive')
DRIVE_OUTPUT_SUBDIR = Path('MyDrive/SpectralBioFinalAcceptA100')

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount(str(DRIVE_MOUNT_POINT))
    print('Drive mounted at', DRIVE_MOUNT_POINT)
else:
    print('Drive mount skipped; outputs stay in the VM unless OUTPUT_ROOT is changed later.')


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/DaviBonetto/SpectralBio.git'
REPO_BRANCH = 'codex/claw4s-rebuild'
REPO_ROOT = Path('/content/Stanford-Claw4s')
BOOTSTRAP_MARKER = REPO_ROOT / '.colab_bootstrap_v3_complete'

if not REPO_ROOT.exists():
    subprocess.check_call(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(REPO_ROOT)])

os.chdir(REPO_ROOT)
subprocess.check_call(['git', 'fetch', 'origin', REPO_BRANCH])
subprocess.check_call(['git', 'checkout', REPO_BRANCH])
src_path = REPO_ROOT / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

if not BOOTSTRAP_MARKER.exists():
    subprocess.check_call([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchvision'])
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', '.', '--no-deps'])
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install',
        'numpy==2.1.3', 'plotly==5.24.1', 'pyyaml==6.0.2', 'scikit-learn==1.5.2',
        'scipy==1.14.1', 'transformers==4.49.0', 'pandas', 'matplotlib'
    ])
    BOOTSTRAP_MARKER.write_text('ok\n', encoding='utf-8')
    print('Dependencies installed. Restarting runtime once...')
    os.kill(os.getpid(), 9)
else:
    print('Bootstrap marker found; skipping reinstall.')


## Plan

- Replace the impression of a narrow support-ranked panel with a visibly broader surface.
- Keep BRCA1 failure analysis principled and predeclared.
- Produce one table for breadth and one figure family for BRCA1 heterogeneity.


In [ ]:
import subprocess
import sys
import pandas as pd

try:
    from spectralbio.supplementary.final_accept_hardening import (
        AcceptHardeningConfig,
        create_accept_hardening_paths,
        run_brca1_failure_analysis,
        run_support_ranked_panel_expansion,
    )
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', '.', '--no-deps'])
    from spectralbio.supplementary.final_accept_hardening import (
        AcceptHardeningConfig,
        create_accept_hardening_paths,
        run_brca1_failure_analysis,
        run_support_ranked_panel_expansion,
    )

TOP_K_GENES = 25
RUN_650M_SHADOW = True
BRCA1_STRATA = ('domain', 'review_status', 'substitution_class', 'confidence')
OVERWRITE = False

OUTPUT_ROOT = (
    DRIVE_MOUNT_POINT / DRIVE_OUTPUT_SUBDIR
    if USE_GOOGLE_DRIVE
    else REPO_ROOT / 'supplementary' / 'final_accept_a100'
)

config = AcceptHardeningConfig(
    stronger_model_names=('facebook/esm2_t33_650M_UR50D', 'facebook/esm2_t36_3B_UR50D'),
    max_additional_genes=23,
    overwrite=OVERWRITE,
)
paths = create_accept_hardening_paths(repo_root=REPO_ROOT, output_root=OUTPUT_ROOT)
print(paths)
print('Top-K genes:', TOP_K_GENES)
print('BRCA1 strata:', BRCA1_STRATA)



In [ ]:
panel25_summary = run_support_ranked_panel_expansion(
    paths=paths,
    config=config,
    top_k=TOP_K_GENES,
    run_650m_shadow=RUN_650M_SHADOW,
)
brca1_failure = run_brca1_failure_analysis(
    paths=paths,
    config=config,
    strata=BRCA1_STRATA,
)
display(pd.DataFrame(panel25_summary['rows']).head())
display(pd.DataFrame(brca1_failure['rows']).head())



In [ ]:
expected_paths = [
    paths.root / 'panel25' / 'support_ranked_top25_summary.csv',
    paths.root / 'panel25' / 'support_ranked_top25_summary.json',
    paths.root / 'brca1_failure' / 'brca1_failure_analysis.csv',
]
for path in expected_paths:
    print(path)


In [ ]:
print('Part 6 scaffold complete. This notebook is reserved for the L4 breadth/failure analysis pass.')
